# Temporal Dynamics of FC: Epileptic vs Non-Epileptic Contacts

Hypothesis: epileptic zones have more variable/bursty connectivity over time.
We use sliding-window correlation (10s windows, 2s step) and compare within-probe.

In [ ]:
import numpy as np
import pandas as pd
import re, sys, warnings
from pathlib import Path
from scipy.stats import mannwhitneyu, spearmanr
from collections import Counter
sys.path.insert(0, 'src')
warnings.filterwarnings('ignore')

# Navigate to project root
import os
while not Path('src/lrg_eegfc').exists():
    os.chdir('..')
print('cwd:', os.getcwd())

from lrg_eegfc.utils.io import load_epileptic_nodes, load_timeseries

In [ ]:
def load_ch(pat):
    p = Path(f'data/stereoeeg_patients/{pat}/channel_labels.csv')
    with open(p) as f:
        first = f.readline().strip()
    skip = 1 if first.lower() == 'label' else 0
    df = pd.read_csv(p, header=None, skiprows=skip)
    return [str(l).strip('"').split(',')[0].strip().replace(' ','') for l in df.iloc[:,0]]

def probe(label):
    m = re.match(r"([A-Za-z]+'?)", label)
    return m.group(1) if m else label

def sliding_window_corr(ts, window_size, step):
    """Compute sliding-window Pearson correlation matrices."""
    N, T = ts.shape
    n_windows = (T - window_size) // step + 1
    matrices = []
    for w in range(n_windows):
        start = w * step
        chunk = ts[:, start:start+window_size]
        mu = chunk.mean(axis=1, keepdims=True)
        std = chunk.std(axis=1, keepdims=True) + 1e-30
        z = (chunk - mu) / std
        C = z @ z.T / window_size
        np.fill_diagonal(C, 0)
        matrices.append(np.abs(C))
    return matrices

In [ ]:
patients = ['Pat_02','Pat_03','Pat_05','Pat_07','Pat_08']
root = Path('data/stereoeeg_patients')
fs_dict = {'Pat_03': 1024}

all_results = {}

for pat in patients:
    epi_set = set(load_epileptic_nodes(pat))
    ch = load_ch(pat)
    N = len(ch)
    epi_mask = np.array([l in epi_set for l in ch])
    
    probes_dict = {}
    for i, label in enumerate(ch):
        p = probe(label)
        if p not in probes_dict:
            probes_dict[p] = {'epi':[], 'non':[]}
        if epi_mask[i]:
            probes_dict[p]['epi'].append(i)
        else:
            probes_dict[p]['non'].append(i)
    mixed = {p:v for p,v in probes_dict.items() if v['epi'] and v['non']}
    if not mixed:
        print(f'\n{"="*70}')
        print(f'{pat}: No mixed probes (epi={epi_mask.sum()}, non={N-epi_mask.sum()})')
        continue
    epi_pool = sum((v['epi'] for v in mixed.values()), [])
    non_pool = sum((v['non'] for v in mixed.values()), [])
    
    fs = fs_dict.get(pat, 2048)
    window_size = int(10 * fs)
    step = int(2 * fs)
    
    print(f'\n{"="*70}')
    print(f'{pat} (fs={fs}, window={window_size/fs:.0f}s, step={step/fs:.0f}s)')
    print(f'  Mixed probes: {list(mixed.keys())}')
    print(f'  Epi contacts in mixed: {len(epi_pool)}, Non-epi in mixed: {len(non_pool)}')
    
    phase = 'rsPre'
    try:
        ts = load_timeseries(pat, phase, root)
    except Exception as e:
        print(f'  Cannot load: {e}')
        continue
    
    if ts.shape[0] != N:
        print(f'  Shape mismatch: ts={ts.shape[0]} vs ch={N}')
        continue
    
    T = ts.shape[1]
    n_win_expected = (T - window_size) // step + 1
    print(f'  T={T} samples = {T/fs:.1f}s, {n_win_expected} windows')
    
    corr_mats = sliding_window_corr(ts, window_size, step)
    n_windows = len(corr_mats)
    
    if n_windows < 5:
        print(f'  Too few windows ({n_windows})')
        continue
    
    # Per-node metrics
    strengths = np.zeros((N, n_windows))
    for w, C in enumerate(corr_mats):
        strengths[:, w] = C.sum(axis=1)
    
    strength_cv = strengths.std(axis=1) / (strengths.mean(axis=1) + 1e-30)
    strength_mean = strengths.mean(axis=1)
    strength_std = strengths.std(axis=1)
    
    # Rank stability
    rank_stability = np.zeros(N)
    for i in range(N):
        conn_profiles = np.array([C[i] for C in corr_mats])
        if n_windows > 2:
            rhos = []
            for w in range(n_windows - 1):
                r, _ = spearmanr(conn_profiles[w], conn_profiles[w+1])
                if not np.isnan(r):
                    rhos.append(r)
            rank_stability[i] = np.mean(rhos) if rhos else 0
    
    # Temporal entropy
    temporal_entropy = np.zeros(N)
    for i in range(N):
        quartiles = np.digitize(strengths[i], np.percentile(strengths[i], [25,50,75]))
        counts = Counter(quartiles)
        probs = np.array([c/n_windows for c in counts.values()])
        temporal_entropy[i] = -np.sum(probs * np.log(probs + 1e-30))
    
    metrics = {
        'strength_CV': strength_cv,
        'strength_mean': strength_mean,
        'strength_std': strength_std,
        'rank_stability': rank_stability,
        'temporal_entropy': temporal_entropy,
    }
    
    pat_results = {}
    print(f'  rsPre results (within-probe, epi vs non-epi):')
    for mname, mvals in metrics.items():
        ve = mvals[epi_pool]
        vn = mvals[non_pool]
        _, pg = mannwhitneyu(ve, vn, alternative='greater')
        _, pl = mannwhitneyu(ve, vn, alternative='less')
        p_best = min(pg, pl)
        d = 'epi>' if pg < pl else 'epi<'
        sig = '***' if p_best < 0.001 else '**' if p_best < 0.01 else '*' if p_best < 0.05 else ''
        print(f'    {mname:20s}: epi={ve.mean():.4f} non={vn.mean():.4f} {d} p={p_best:.4f} {sig}')
        pat_results[mname] = {'epi_mean': ve.mean(), 'non_mean': vn.mean(), 'direction': d, 'p': p_best, 'sig': sig}
    
    all_results[pat] = pat_results

print('\nDone.')

In [ ]:
# Summary table: which metrics show consistent within-probe differences across patients?
print('\n' + '='*80)
print('CROSS-PATIENT SUMMARY')
print('='*80)

metric_names = ['strength_CV', 'strength_mean', 'strength_std', 'rank_stability', 'temporal_entropy']

for mname in metric_names:
    print(f'\n--- {mname} ---')
    directions = []
    sig_count = 0
    consistent_dir = True
    first_dir = None
    for pat, res in all_results.items():
        if mname in res:
            r = res[mname]
            directions.append(r['direction'])
            if r['sig']:
                sig_count += 1
            if first_dir is None:
                first_dir = r['direction']
            elif r['direction'] != first_dir:
                consistent_dir = False
            print(f'  {pat}: {r["direction"]} epi={r["epi_mean"]:.4f} non={r["non_mean"]:.4f} p={r["p"]:.4f} {r["sig"]}')
    
    n_patients = len(directions)
    consistency = 'CONSISTENT' if consistent_dir and n_patients > 1 else 'MIXED'
    print(f'  => Direction: {consistency} ({first_dir if consistent_dir else "varies"}), Significant in {sig_count}/{n_patients} patients')